In [ ]:
import pandas as pd
import numpy as np
import os
import json 

os.getcwd()

In [ ]:
package_data = pd.read_csv('longitudinal_study_package_criteria_october.csv')
package_data['githubrepo'] = 'https://github.com/' + package_data['ProjectName']

# All repos in the package list — release data hasn't been collected yet
urls_to_get_df = package_data[['githubrepo']].drop_duplicates().reset_index(drop=True)

In [ ]:
GITHUB_TOKENS = [
   xxxx
]

## Collecting release and tag information from GitHub REST API. 

In [ ]:
import pandas as pd
import requests
from typing import Dict, List, Optional
import time
import re

def extract_repo_info(github_url: str) -> tuple:
    """
    Extract owner and repo name from GitHub URL.
    
    Args:
        github_url: GitHub repository URL
    
    Returns:
        Tuple of (owner, repo_name) or (None, None) if parsing fails
    """
    if pd.isna(github_url) or github_url == '':
        return None, None
    
    github_url = str(github_url).strip()
    github_url = re.sub(r'\.git$', '', github_url)
    
    match = re.search(r'github\.com/([^/]+)/([^/\s]+)', github_url)
    if match:
        return match.group(1), match.group(2)
    
    if '/' in github_url and 'github.com' not in github_url:
        parts = github_url.split('/')
        if len(parts) >= 2:
            return parts[0], parts[1]
    
    return None, None


def get_all_tags_and_releases(repo_owner: str, repo_name: str, github_token: Optional[str] = None) -> Dict:
    """
    Fetch all tags and releases for a repository.
    
    Args:
        repo_owner: Repository owner
        repo_name: Repository name
        github_token: GitHub personal access token
    
    Returns:
        Dictionary with tags, releases, and metadata
    """
    headers = {}
    if github_token:
        headers['Authorization'] = f'token {github_token}'
    
    result = {
        'repo_owner': repo_owner,
        'repo_name': repo_name,
        'tags': [],
        'releases': [],
        'status': 'success',
        'rate_limited': False
    }
    
    # Fetch all tags (paginated)
    try:
        page = 1
        while True:
            tags_url = f'https://api.github.com/repos/{repo_owner}/{repo_name}/tags?per_page=100&page={page}'
            response = requests.get(tags_url, headers=headers, timeout=10)
            
            if response.status_code == 200:
                tags_data = response.json()
                if not tags_data:  # No more tags
                    break
                
                for tag in tags_data:
                    result['tags'].append({
                        'name': tag['name'],
                        'commit_sha': tag['commit']['sha']
                    })
                
                page += 1
                
                # Check if there are more pages
                if 'Link' not in response.headers or 'rel="next"' not in response.headers['Link']:
                    break
                    
            elif response.status_code == 404:
                result['status'] = 'repo_not_found'
                break
            elif response.status_code == 403:
                if 'X-RateLimit-Remaining' in response.headers:
                    remaining = response.headers.get('X-RateLimit-Remaining', '0')
                    if remaining == '0':
                        result['status'] = 'rate_limited'
                        result['rate_limited'] = True
                        return result
                break
            else:
                result['status'] = f'error_{response.status_code}'
                break
                
    except Exception as e:
        result['status'] = f'exception_tags_{str(e)}'
    
    # Fetch all releases (paginated)
    try:
        page = 1
        while True:
            releases_url = f'https://api.github.com/repos/{repo_owner}/{repo_name}/releases?per_page=100&page={page}'
            response = requests.get(releases_url, headers=headers, timeout=10)
            
            if response.status_code == 200:
                releases_data = response.json()
                if not releases_data:  # No more releases
                    break
                
                for release in releases_data:
                    result['releases'].append({
                        'tag_name': release['tag_name'],
                        'name': release.get('name', ''),
                        'commit_sha': release.get('target_commitish', ''),
                        'published_at': release.get('published_at', '')
                    })
                
                page += 1
                
                # Check if there are more pages
                if 'Link' not in response.headers or 'rel="next"' not in response.headers['Link']:
                    break
                    
            elif response.status_code == 404:
                # Repo might not have releases, which is fine
                break
            elif response.status_code == 403:
                if 'X-RateLimit-Remaining' in response.headers:
                    remaining = response.headers.get('X-RateLimit-Remaining', '0')
                    if remaining == '0':
                        result['status'] = 'rate_limited'
                        result['rate_limited'] = True
                        return result
                break
            else:
                break
                
    except Exception as e:
        result['status'] = f'exception_releases_{str(e)}'
    
    return result

def fetch_all_repo_tags(df: pd.DataFrame,
                        output_file: str = 'all_repo_tags.csv',
                        github_tokens: Optional[List[str]] = None,
                        delay: float = 0.5,
                        save_every: int = 100) -> pd.DataFrame:
    """
    Fetch all tags and releases for unique repositories in the dataframe.
    
    Args:
        df: DataFrame with 'githubrepo' column
        output_file: Output CSV file path
        github_tokens: List of GitHub tokens to rotate
        delay: Delay between requests
        save_every: Save progress every N repositories
    
    Returns:
        DataFrame with all tags and releases per repository
    """
    import json
    import os

    # Get unique repositories
    unique_repos = df['githubrepo'].unique()
    total_repos = len(unique_repos)
    
    # Token rotation setup
    current_token_index = 0
    token_stats = {}
    
    if github_tokens:
        print(f"Using {len(github_tokens)} GitHub tokens for rotation")
        for i in range(len(github_tokens)):
            token_stats[i] = {'requests': 0, 'rate_limited': False}
    else:
        github_tokens = [None]
        print("No GitHub tokens provided - using unauthenticated requests")
    
    def get_next_token():
        """Get next available token"""
        nonlocal current_token_index
        
        if len(github_tokens) == 1 and github_tokens[0] is None:
            return None
        
        attempts = 0
        while attempts < len(github_tokens):
            token_index = current_token_index % len(github_tokens)
            
            if not token_stats[token_index]['rate_limited']:
                current_token_index = token_index
                return github_tokens[token_index]
            
            current_token_index += 1
            attempts += 1
        
        print("\n⚠️ All tokens rate limited! Resetting...")
        for idx in token_stats:
            token_stats[idx]['rate_limited'] = False
        
        current_token_index = 0
        return github_tokens[0]
    
    results = []
    last_save_index = 0  # <-- Track last saved index
    
    for count, repo_url in enumerate(unique_repos, start=1):
        repo_owner, repo_name = extract_repo_info(repo_url)
        
        if not repo_owner or not repo_name:
            print(f"Repo {count}/{total_repos}: Could not parse '{repo_url}'")
            results.append({
                'githubrepo': repo_url,
                'repo_owner': 'N/A',
                'repo_name': 'N/A',
                'tags': '[]',
                'releases': '[]',
                'num_tags': 0,
                'num_releases': 0,
                'status': 'invalid_url'
            })
            continue
        
        # Get current token
        current_token = get_next_token()
        token_index = current_token_index
        
        if count % 10 == 0:
            token_info = f"Token {token_index + 1}/{len(github_tokens)}" if github_tokens[0] else "No token"
            print(f"Repo {count}/{total_repos} [{token_info}]: Fetching {repo_owner}/{repo_name}...")
        
        # Fetch all tags and releases
        repo_data = get_all_tags_and_releases(repo_owner, repo_name, current_token)
        
        # Track token usage
        if github_tokens[0] is not None:
            token_stats[token_index]['requests'] += 1
        
        # Handle rate limiting
        if repo_data.get('rate_limited', False):
            print(f"\n⚠️ Token {token_index + 1} RATE LIMITED (after {token_stats[token_index]['requests']} requests)")
            token_stats[token_index]['rate_limited'] = True
            
            current_token_index += 1
            current_token = get_next_token()
            token_index = current_token_index
            
            print(f"🔄 Switching to Token {token_index + 1}, retrying...\n")
            
            repo_data = get_all_tags_and_releases(repo_owner, repo_name, current_token)
            if github_tokens[0] is not None:
                token_stats[token_index]['requests'] += 1
        
        # Append new results
        results.append({
            'githubrepo': repo_url,
            'repo_owner': repo_owner,
            'repo_name': repo_name,
            'tags': json.dumps(repo_data['tags']),
            'releases': json.dumps(repo_data['releases']),
            'num_tags': len(repo_data['tags']),
            'num_releases': len(repo_data['releases']),
            'status': repo_data['status']
        })
        
        time.sleep(delay)
        
        # Save progress every `save_every` repos
        if count % save_every == 0:
            print(f"\n{'='*60}")
            print(f"SAVING: Processed {count}/{total_repos} repositories")
            
            if github_tokens[0] is not None:
                print("\nToken usage:")
                for tid, stats in token_stats.items():
                    status = "RATE LIMITED" if stats['rate_limited'] else "Active"
                    print(f"  Token {tid + 1}: {stats['requests']} requests - {status}")
            
            print(f"{'='*60}\n")
            
            # Only append new batch
            new_rows = results[last_save_index:count]
            batch_df = pd.DataFrame(new_rows)
            write_header = not os.path.exists(output_file)
            batch_df.to_csv(output_file, mode='a', header=write_header, index=False)
            print(f"Saved to '{output_file}'\n")
            
            last_save_index = count  # update last saved index
    
    # Final save (append only remaining new rows)
    remaining_rows = results[last_save_index:]
    if remaining_rows:
        final_df = pd.DataFrame(remaining_rows)
        write_header = not os.path.exists(output_file)
        final_df.to_csv(output_file, mode='a', header=write_header, index=False)
    
    print(f"\n{'='*60}")
    print(f"COMPLETE: Processed all {total_repos} repositories")
    print(f"{'='*60}\n")
    print(f"Final results saved to '{output_file}'")
    
    return pd.DataFrame(results)

In [ ]:
# Then run the function
all_tags_df = fetch_all_repo_tags(
    urls_to_get_df,
    output_file='all_repo_tags_new_data.csv',
    github_tokens=GITHUB_TOKENS,
    delay=0.5,
    save_every=50
)